# Process Images Multimodal Data with 

The app data is not limited to the unstructured text. It also includes food recipes and user visit histories that reference images, capturing visual information that text alone cannot fully describe, such as presentation style, ingredients, and portion details. To unlock this information and integrate it with the original databases, I am useing multimodal GenAI capabilities to convert images into descriptive text and enrich the structured knowledge.

**Objectives**
- Load food recipe and user visit data containing image references
- Use a multimodal foundation model to generate textual captions for food images
- Integrate the generated image captions into the existing structured JSON data
- Produce an enriched multimodal JSON knowledge file for downstream applications

**Task**
- Apply image captioning to generate descriptive text for food images
- Align and merge image captions with the corresponding recipe and restaurant JSON records
- Extend the existing structured knowledge base with multimodal context while preserving a consistent schema


### Importing required libraries

In [2]:
import numpy as np
import json
import os
import matplotlib.pyplot as plt
from PIL import Image
from openai import OpenAI
import time
import zipfile

def warn(*args, **kwargs):
    pass
import warnings
warnings.warn=warn
warnings.filterwarnings("ignore")

# Fetch the data file
!curl 0 https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/hpTjb6liKBLVHQK0UgMi5A/Recipes.json  
!curl 0 https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/fQUs9wQ6aB6ts6fmkD2V2w/Synthetic-User-Reviews.json  
!curl 0 https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/5_Rr6ohviItzucyWk6nkrw/synthetic-recipe-images.zip

*Note: Use wget for linux system*

In [ ]:
# The file contains the images of the food to extract the recipe and the presentation style

with zipfile.ZipFile("synthetic-recipe-images.zip", 'r') as zip_ref:
    zip_ref.extractall()

## Preprocess the food reciepe data

### Explore the food recipe JSON File and its images

Before applying any multimodal processing, let us understand the structure and contents of the data. In this step, I am loading the food recipe JSON file and examine its fields, including textual metadata and image references.

Also inspect a sample of the associated food images to understand the visual information they contain, such as ingredients, presentation style, and portion size. 
This exploration step helps you identify how the image data aligns with recipe metadata and prepares you for integrating image captions into the existing JSON structure in later steps.

This will provide us with clear understanding of how the recipe data and images are organized, and how they relate to the structured restaurant knowledge.

In [7]:
# Load the recipe json data file
with open("Recipes.json", "r") as file:
    recipe_data = json.load(file)
# Review the key value pair of the first recipe
print(f"First recipe \n {recipe_data[0]}")
# Review the image of the first recipe
with Image.open("synthetic_recipe_images/recipe1.png") as img:
    img.show()

First recipe 
 {'id': 1, 'name': 'Classic Margherita Pizza', 'cuisine': 'Italian', 'servings': 2, 'prep_time': '20 mins', 'cook_time': '15 mins', 'total_time': '35 mins', 'ingredients': ['1 pizza dough ball (about 260g)', '2 tablespoons tomato sauce', '1 cup shredded mozzarella cheese', '6-8 fresh basil leaves', '1 tablespoon olive oil', 'pinch of salt', 'optional: grated parmesan'], 'directions': ['Preheat oven to 250Â°C (482Â°F) with a baking stone if available.', 'Roll out dough to a 12-inch circle on a floured surface.', 'Spread tomato sauce evenly over dough, leaving a 1/2-inch border.', 'Scatter mozzarella cheese over sauce.', 'Bake on the stone for 10-12 minutes until crust is crisp and cheese melts.', 'Finish with basil leaves, drizzle with olive oil, and season with salt and optional parmesan.', 'Slice and serve immediately.']}


### Define the vision LLM with LLaMA

In [12]:
import base64
def vision_llm(system_msg, prompt_txt, image_path):
    client = OpenAI(
        base_url="http://localhost:11434/v1",
        api_key= "ollama"
    )
    model_id = "llama3.2"#'meta-llama/llama-4-maverick-17b-128e-instruct-fp8' 
    # Encode the image into base64 string
    with Image.open(image_path) as img:
        img = img.convert("RGB")
        img_bytes = img.tobytes()
        base64_str = base64.b64encode(img_bytes).decode('utf-8') 
    # Define the message for the model
    messages = [
        {"role": 'system', "content" : system_msg},
        {"role": "user", "content": prompt_txt}
        ]
    # Defining the model response
    response = client.chat.completions.create(
        messages=messages,
        model=model_id)
    #Generate response from the model
    output_text = response.choices[0].message.content
    return output_text

### Design and validate prompts for the vision LLM

To ensure the vision LLM generates useful and consistent image captions, careful prompt design is essential. In this step, you will create a prompt that instructs the model to focus on relevant visual details, such as ingredients, cooking style, and presentation, while avoiding unnecessary or speculative information.

In [10]:
def image_caption_prompt_template(food_name):
    # food name : food name of the recipe

    #Designe the prompt
    image_caption_system_msg = f"""
    You are an expert food photographer and culinary reviewer. Your task is to generate descriptive, precise, and objective captions for food images.

    Follow these strict rules when generating captions:
    1. Focus on Observable Visual Details:
       - Key Ingredients: List visible components (e.g., garnishes, toppings, primary proteins, vegetables, sauces).
       - Cooking & Prep Style: Describe observable texture and cooking indicators (e.g., charred edges, golden-brown crust, fresh raw cuts, melted cheese, glossy glaze).
       - Presentation & Plating: Specify the dishware (e.g., white ceramic plate, cast-iron skillet, clear glass tumbler) and layout (e.g., stacked, drizzled, garnished on top).
    
    2. Avoid Unnecessary & Speculative Information:
       - Do NOT guess unobservable flavors, spice levels, origin, recipe steps, or exact names unless visually obvious (e.g., do not write "spicy authentic Italian pasta", write "penne pasta tossed in red tomato sauce garnished with basil").
       - Avoid subjective fluff (e.g., "delicious," "mouth-watering," "tasty," "perfectly cooked").
       - Do not describe background objects unless they are directly relevant to the food context (e.g., a side dish or drink).
    
    3. Formatting:
       - Start with a single direct summary sentence naming the primary subject.
       - Follow with a structured breakdown: Main Dish, Key Ingredients & Visual Textures, and Presentation. 
    """
    image_caption_prompt_txt = (
    f"Examine this image of {food_name}. Generate a detailed visual caption focusing "
    f"strictly on its visible ingredients, cooking/preparation style, and plating presentation."
    )

    return image_caption_system_msg, image_caption_prompt_txt

### Test the prompt on the first recipe

In [13]:
# Get the prompt with food name of the first recipe
image_1 = "synthetic_recipe_images/recipe1.png"
food_name_1 = recipe_data[0]['name']
image_caption_system_msg, image_caption_prompt_txt = image_caption_prompt_template(food_name=food_name_1)
response = vision_llm(
    system_msg=image_caption_system_msg, 
    prompt_txt=image_caption_prompt_txt,
    image_path=image_1)
print(response)

"Classic Margherita Pizza"

**Main Dish:** Classic Margherita Pizza, a traditional Neapolitan-style pizza.

**Key Ingredients & Visual Textures:** 
- Fresh mozzarella cheese, sliced and distributed evenly across the pizza surface.
- San Marzano tomatoes, crushed and spread in a symmetrical layer.
- Fresh basil leaves, chopped and sprinkled artfully in a decorative pattern.

**Presentation:** 
- The pizza is served on a white ceramic plate at a slight angle, creating a dynamic composition.
- The crust is slightly charred and rounded, with a shallow golden-brown crust along the edges.
- The dish is framed by a thin border of white ceramic, creating a clean and minimalist aesthetic.


### Caption all images and augment the data

In [14]:
# File directory
image_dir = "synthetic_recipe_images"
# Get caption of each image in the dataset and add to the JSON file
for i in range(len(recipe_data)):
    if (i+1)%10==0:
        print(f'{i+1} out of {len(recipe_data)} is done \n')
    # Get the prompt
    food_name = recipe_data[i]['name']
    sys_mes, pmt_txt = image_caption_prompt_template(food_name=food_name)
    # Get the response with the prompt
    image_filename = recipe_data[i].get('image_file', f'recipe{i+1}.png')
    img_path = os.path.join(image_dir, image_filename)
    response = vision_llm(
        system_msg=sys_mes,
        prompt_txt=pmt_txt,
        image_path= img_path
    )
    # Save the response as another item in the recipe data
    recipe_data[i]["image_description"] = response
print("All image processed")

10 out of 109 is done
20 out of 109 is done
30 out of 109 is done
40 out of 109 is done
50 out of 109 is done
60 out of 109 is done
70 out of 109 is done
80 out of 109 is done
90 out of 109 is done
100 out of 109 is done
All image processed


### Save the image-caption-augmented recipe data

In [16]:
filename = 'augmented_food_recipe.json'
with open(filename, 'w', encoding='utf-8') as f:
    json.dump(recipe_data, f, indent=4)

### Preprocess the user visit history